<a href="https://colab.research.google.com/github/Mairrch/CIAP---Sentiment-Analysis/blob/comparacion-con-transformer/03_transformer_comparacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Comparación con Transformer (DistilBERT)
Fine-tuning de un modelo preentrenado en inglés (**DistilBERT**, versión comprimida de BERT) para compararlo contra el baseline TF-IDF + Regresión Logística del notebook 02. El dataset está en inglés, así que se usa un modelo en inglés.

**Pregunta central:** ¿cuánto mejora el recall de la clase *negativo (0)* (baseline: 0.66) y a qué costo en tiempo de cómputo?

> **Antes de correr:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU (T4)**.

### 1. Entorno (Colab)
Clono el repo para que las rutas relativas (`../data/train.csv`) funcionen igual que en local.
Cambiar `REPO_URL` y `REPO_NAME`. Si el repo es privado, guardar un token de GitHub en *Secrets* de Colab (ícono de llave) con el nombre `GH_TOKEN`.

In [ ]:
import os

REPO_URL = "https://github.com/USUARIO/REPO.git"   # <- CAMBIAR
REPO_NAME = "REPO"                                  # <- CAMBIAR

# Si el repo es privado, descomentar estas 3 líneas:
# from google.colab import userdata
# token = userdata.get("GH_TOKEN")
# REPO_URL = REPO_URL.replace("https://", f"https://{token}@")

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
%cd {REPO_NAME}/notebooks
!pip install -q transformers datasets accelerate

In [ ]:
import time
import json
import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, f1_score, accuracy_score,
                             recall_score, confusion_matrix, ConfusionMatrixDisplay)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

SEED = 42
set_seed(SEED)

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("OJO: sin GPU el entrenamiento tardará muchísimo. Activa la GPU en el entorno de ejecución.")

### 2. Configuración
Todo lo ajustable está aquí. Con `SAMPLE = 20000` se hace una **prueba rápida** (unos minutos) para verificar que todo corre antes de lanzar el entrenamiento completo (`SAMPLE = None`).

In [ ]:
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"

# Modelos en inglés sugeridos (cambiar solo esta línea para probar otro):
#   "distilbert-base-uncased"  -> rápido y liviano (por defecto)
#   "bert-base-uncased"        -> BERT original, referencia de calidad
#   "roberta-base"             -> suele rendir mejor que BERT
MODEL_NAME = "distilbert-base-uncased"
MODEL_TAG = MODEL_NAME.split("/")[-1]   # se usa para nombrar los archivos de salida
NUM_LABELS = 3
MAX_LEN = 128
EPOCHS = 3
BATCH_SIZE = 32
LR = 2e-5
SAMPLE = None   # p. ej. 20000 para prueba rápida; None = datos completos

Path(MODELS_DIR).mkdir(exist_ok=True)
Path(RESULTS_DIR).mkdir(exist_ok=True)

### 3. Carga de datos
Mismos `train.csv` y `test.csv` del baseline, para que la comparación sea justa.
Separo una **validación** (5% del train) para elegir el mejor epoch. El `test` queda intocado hasta la evaluación final.

In [ ]:
train_full = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

train_full["Comment"] = train_full["Comment"].fillna("")
test["Comment"] = test["Comment"].fillna("")

if SAMPLE:
    train_full = train_full.sample(SAMPLE, random_state=SEED)
    test = test.sample(min(SAMPLE // 4, len(test)), random_state=SEED)

train, val = train_test_split(train_full, test_size=0.05, stratify=train_full["Sentiment"], random_state=SEED)

print("train:", train.shape, "| val:", val.shape, "| test:", test.shape)
print(train["Sentiment"].value_counts(normalize=True).sort_index().round(3))

### 4. Tokenización
Reviso cuántos tokens suelen tener los comentarios para justificar `MAX_LEN`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

muestra = train["Comment"].sample(min(5000, len(train)), random_state=SEED).tolist()
lens = [len(tokenizer.encode(t, truncation=False)) for t in muestra]
print("Percentiles de longitud (tokens):", {p: int(np.percentile(lens, p)) for p in [50, 90, 95, 99]})
print(f"% de comentarios que superan MAX_LEN={MAX_LEN}:", round(np.mean(np.array(lens) > MAX_LEN) * 100, 2))

In [ ]:
def to_hf(df):
    ds = Dataset.from_dict({"text": df["Comment"].tolist(), "labels": df["Sentiment"].astype(int).tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = to_hf(train), to_hf(val), to_hf(test)
collator = DataCollatorWithPadding(tokenizer=tokenizer)   # padding dinámico: más rápido

### 5. Fine-tuning
La métrica para elegir el mejor epoch es **Macro F1** (la misma del baseline).

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

args = TrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=200,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

t0 = time.perf_counter()
trainer.train()
train_time = time.perf_counter() - t0
print(f"\nTiempo de entrenamiento: {train_time:.1f} s ({train_time/60:.1f} min)")

### 6. Evaluación en `test` y tiempo de inferencia

In [ ]:
t0 = time.perf_counter()
out = trainer.predict(test_ds)
infer_time = time.perf_counter() - t0

y_test = test["Sentiment"].astype(int).values
y_pred_tr = np.argmax(out.predictions, axis=-1)

target_names = ["negativo (0)", "neutral (1)", "positivo (2)"]
print("Reporte de Clasificación (Transformer):")
print(classification_report(y_test, y_pred_tr, target_names=target_names, digits=2))

acc_tr = accuracy_score(y_test, y_pred_tr)
f1_tr = f1_score(y_test, y_pred_tr, average="macro")
ms_tr = infer_time / len(test) * 1000
print(f"Accuracy: {acc_tr:.4f} | Macro F1: {f1_tr:.4f}")
print(f"Inferencia: {infer_time:.2f} s ({ms_tr:.4f} ms/comentario) en {'GPU' if torch.cuda.is_available() else 'CPU'}")

### 7. Baseline sobre el mismo `test`
Cargo los artefactos del notebook 02 y los evalúo sobre **exactamente las mismas filas** de `test` (importante si usaste `SAMPLE`). Si no se pueden cargar, uso las métricas reportadas en el notebook 02.

In [ ]:
baseline_ok = False
try:
    vec = joblib.load(f"{MODELS_DIR}/tfidf_vectorizer.joblib")
    lr = joblib.load(f"{MODELS_DIR}/logreg_baseline.joblib")
    t0 = time.perf_counter()
    y_pred_bl = lr.predict(vec.transform(test["Comment"]))
    infer_bl = time.perf_counter() - t0
    baseline_ok = True
    print("Baseline cargado y evaluado sobre el mismo test.")
except Exception as e:
    print("No se pudo cargar el baseline:", repr(e))
    print("Se usarán las métricas reportadas en el notebook 02 (solo comparación global).")

if baseline_ok:
    acc_bl = accuracy_score(y_test, y_pred_bl)
    f1_bl = f1_score(y_test, y_pred_bl, average="macro")
    rec_neg_bl = recall_score(y_test, y_pred_bl, labels=[0], average=None)[0]
    ms_bl = infer_bl / len(test) * 1000
else:
    acc_bl, f1_bl, rec_neg_bl, ms_bl = 0.7964, 0.7825, 0.66, 0.0706 / 41690 * 1000
    y_pred_bl = None

### 8. Tabla comparativa

In [ ]:
rec_neg_tr = recall_score(y_test, y_pred_tr, labels=[0], average=None)[0]

comp = pd.DataFrame({
    "Modelo": ["TF-IDF + LogReg", f"{MODEL_TAG} (fine-tuned)"],
    "Accuracy": [acc_bl, acc_tr],
    "Macro F1": [f1_bl, f1_tr],
    "Recall negativo": [rec_neg_bl, rec_neg_tr],
    "ms/comentario (inferencia)": [ms_bl, ms_tr],
}).round(4)
comp

In [ ]:
if y_pred_bl is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, yp, titulo in zip(axes, [y_pred_bl, y_pred_tr], ["Baseline", MODEL_TAG]):
        cm = confusion_matrix(y_test, yp, normalize="true")
        ConfusionMatrixDisplay(cm, display_labels=["neg", "neu", "pos"]).plot(ax=ax, colorbar=False, values_format=".2f")
        ax.set_title(f"{titulo} (normalizada por fila)")
    plt.tight_layout()
    plt.show()

### 9. Análisis de errores en negativos
¿Qué negativos se siguen escapando? ¿Cuáles corrigió el Transformer respecto al baseline? Esto alimenta el notebook 04.

In [ ]:
res = test[["Comment"]].copy()
res["y_true"] = y_test
res["y_pred_transformer"] = y_pred_tr
if y_pred_bl is not None:
    res["y_pred_baseline"] = y_pred_bl

neg = res[res["y_true"] == 0]
pd.set_option("display.max_colwidth", 200)

print("== Negativos que el Transformer sigue fallando ==")
display(neg[neg["y_pred_transformer"] != 0].sample(min(15, (neg["y_pred_transformer"] != 0).sum()), random_state=SEED))

if y_pred_bl is not None:
    corregidos = neg[(neg["y_pred_baseline"] != 0) & (neg["y_pred_transformer"] == 0)]
    print(f"\n== Negativos que el baseline fallaba y el Transformer corrige ({len(corregidos)} en total) ==")
    display(corregidos.sample(min(10, len(corregidos)), random_state=SEED))

### 10. Guardar resultados
Las predicciones y métricas son livianas y van al repo (`results/`), con el nombre del modelo en el archivo para no pisar corridas anteriores. El modelo pesa cientos de MB: **no subirlo a GitHub**, guardarlo en Drive o en Hugging Face Hub.

In [ ]:
res.to_csv(f"{RESULTS_DIR}/predicciones_test_{MODEL_TAG}.csv", index=False)

metrics = {
    "modelo": MODEL_NAME, "max_len": MAX_LEN, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
    "n_test": int(len(test)), "accuracy": float(acc_tr), "macro_f1": float(f1_tr),
    "recall_negativo": float(rec_neg_tr), "train_time_s": float(train_time),
    "infer_ms_per_comment": float(ms_tr),
    "device_infer": "GPU" if torch.cuda.is_available() else "CPU",
}
with open(f"{RESULTS_DIR}/metricas_{MODEL_TAG}.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print("Guardado en", RESULTS_DIR)

In [ ]:
# OPCIONAL: guardar el modelo en tu Drive (cada persona en el suyo)
# from google.colab import drive
# drive.mount("/content/drive")
# trainer.save_model(f"/content/drive/MyDrive/CIAP/{MODEL_TAG}_sentiment")
# tokenizer.save_pretrained(f"/content/drive/MyDrive/CIAP/{MODEL_TAG}_sentiment")

### 11. Tabla consolidada de todos los modelos corridos
Si corres este notebook con varios `MODEL_NAME`, aquí se juntan todos los resultados guardados en `results/`.

In [ ]:
import glob
filas = []
for p in sorted(glob.glob(f"{RESULTS_DIR}/metricas_*.json")):
    with open(p) as f:
        m = json.load(f)
    filas.append({"Modelo": m["modelo"], "Accuracy": m["accuracy"], "Macro F1": m["macro_f1"],
                  "Recall negativo": m["recall_negativo"], "Entrenamiento (min)": m["train_time_s"] / 60,
                  "ms/comentario": m["infer_ms_per_comment"], "Dispositivo": m["device_infer"]})
pd.concat([pd.DataFrame([{"Modelo": "TF-IDF + LogReg (baseline)", "Accuracy": acc_bl, "Macro F1": f1_bl,
                          "Recall negativo": rec_neg_bl, "ms/comentario": ms_bl, "Dispositivo": "CPU"}]),
           pd.DataFrame(filas)], ignore_index=True).round(4)

### 12. Conclusiones (completar con los resultados reales)
- **Calidad:** Macro F1 pasó de 0.78 a ___; el recall de negativos de 0.66 a ___.
- **Costo:** entrenamiento ___ min en GPU (baseline: segundos en CPU); inferencia ___ ms/comentario vs ___ ms.
- **Errores que persisten:** (ironía, negaciones, neutrales ambiguos...) según la sección 9.
- **Siguiente paso:** usar `results/predicciones_test.csv` (negativos reales y predichos) para el notebook 04 de tópicos.